# SpMV scaling: setup strips, synced timing, weak scaling to the full wafer (2026-10-09)

The 2026-10-09 CS-3 window: 65K setup-strips A/B synced on-wafer timing, and weak scaling at ~200 NNZ/PE, d=1e-3,
from 1,024 to 857,010 PEs (n=3 per point), all with the new defaults: valtask, column mode, col-major setup, 16 channels
(clamped to the grid height), 32 setup strips. Wall-time comparisons reach back to the 10-07 and 10-08 runs. Only barrier-closed
times count. Predictions were written down before the runs; where a model was fitted afterwards, the text says so.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import lib
lib.setup_plots()
pd.set_option("display.width", 200, "display.max_columns", 30)

In [ ]:
F = {"weak": "2026-10-09_weak-scaling.csv", "d56": "2026-10-09_dan56.csv", "strips": "2026-10-09_setup-strips.csv",
     "xsys": "2026-10-09_xsys-check.csv", "fw": "2026-10-08_fullwafer.csv",
     "old": "2026-10-07_barrier-remeasure.csv", "s1": "2026-10-07_s1-column-matrix.csv"}
raw = {k: lib.load_spmv([v]) for k, v in F.items()}

def ok(d):
    """Validated, SUCCESS, barrier-closed stream time present."""
    if d.empty:
        return d
    keep = d.validation_passed.astype(bool) & d.exit_reason.eq("SUCCESS") & (d.stream_time_corrected.fillna(0) > 0)
    return d[keep].copy()

D = {k: ok(v) for k, v in raw.items()}
for k, v in raw.items():
    print(f"{F[k]:38s} {len(v):3d} rows, {len(D[k]):3d} usable")
for d in D.values():
    if not d.empty:
        d["sync"] = d.get("sync_timing", pd.Series(False, index=d.index)).astype(str).eq("True")
        d["strips"] = d.get("setup_strips", pd.Series(1, index=d.index)).fillna(1).astype(int)
        d["stream_ms"] = d.stream_time_corrected * 1e3
        d["setup_ms"] = d.setup_h2d_time_corrected * 1e3
        d["stream_GBps"] = d.h2d_bytes_total / d.stream_time_corrected / 1e9
        d["setup_GBps"] = d.setup_h2d_bytes / d.setup_h2d_time_corrected / 1e9
# The 65K weak-scaling point is the flag-off synced-timing runs: identical config, reuse noted in commit 0cbc1c4.
ws = pd.concat([D["weak"], D["d56"][~D["d56"].sync]], ignore_index=True).sort_values("num_pes")
# Systems are not in these CSVs; from the run reports: system A for every 10-09 run up to 428K run 1,
# system B for 428K runs 2-3, the full wafer and the 65K cross-check.

---
## 1. Setup copy in 32 narrow strips vs one rectangle (65K PEs)

### Hypothesis

**Prediction (plan page, 2026-10-09, before the runs):** M1/M4 reached 10.3 GB/s at 16 channels with narrow strips issued
serially, against 6.3–7.2 GB/s for one rectangle in S1. Expected 8–9.5 GB/s for 32 strips (65K has only 4.6 KB/PE, so per-strip
cost should show), hoped ~10. Stream time should not move: only the setup copy changes.
**Falsifier:** 32 strips no faster than 1, or stream time changing by more than run-to-run noise.

### Result

In [ ]:
s = pd.concat([D["strips"].assign(system="system A"), D["xsys"].assign(system="system B")], ignore_index=True)
if s.empty:
    lib.no_data("2026-10-09_setup-strips.csv")
else:
    display(lib.summarize(s, ["strips", "system"], "setup_ms", 1, 1).rename(columns=lambda c: c + " ms" if c in ("min", "median", "max") else c))
    display(lib.summarize(s, ["strips", "system"], "setup_GBps", 1, 2).rename(columns=lambda c: c + " GB/s" if c in ("min", "median", "max") else c))
    display(lib.summarize(s, ["strips"], "stream_ms", 1, 1).rename(columns=lambda c: c + " stream ms" if c in ("min", "median", "max") else c))
    m = s[s.system == "system A"].groupby("strips").setup_ms.median()
    print(f"speed-up 1 -> 32 strips (median, system A): {m[1] / m[32]:.2f}x")
    fig, axs = lib.panels(1, height=4.6)
    ax = axs[0]
    for i, (k, g) in enumerate(s.groupby("strips")):
        st = lib.style("index", i)
        ax.scatter(g.strips, g.setup_GBps, s=60, color=st["color"], marker=st["marker"], edgecolors="white", linewidths=0.6,
                   label=f"{k} strip{'s' if k > 1 else ''}", zorder=3)
        m76 = g[g.system == "system A"].setup_GBps.median()
        ax.annotate(f"median {m76:.1f} GB/s (system A)", (k, m76), xytext=(10, 0),
                    textcoords="offset points", color=st["color"], va="center")
    lib.draw_claims(ax, 16)
    lib.finish(ax, "setup strips (count)", "setup copy, barrier-closed (GB/s)", "65,536 PEs, 302 MB setup, 16 channels: every run",
               logx=True, xticks=[1, 32])
    ax.set_ylim(0, 16)
    lib.place_claims(fig); plt.show()

### What that means

*(Written 2026-10-13 from the rows above.)*
- **32 strips set up 1.48× faster** than one rectangle: 27.7 vs 41.0 ms median, 10.9 vs 7.4 GB/s. Every 32-strip run beats every 1-strip run.
  That beats the hope, and is at or above the 10.3 GB/s the microbenchmarks reported.
- The single system B run (26.0 ms) falls inside the system A range, so there's no sign of a system effect.
- Stream time doesn't move (46.5 vs 45.9 ms medians), as predicted.
- **Not established:** why strips help. Likely it's more transfers in flight across the 16 channels. That was not isolated here.

### What the vendor says

**Silent on strips.** The guide's only shape advice is about order: [SDK]:313 "Column-major delivers better bandwidth than
row-major." **Context:** [BW]:18–19 "The maximum bandwidth of a single channel is around 7Gbps", i.e. 0.875 GB/s per channel,
14 GB/s at 16 channels (red line). 32 strips reach 78% of that comment's 16-channel figure.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 2. Does an independent on-wafer clock agree with the host barrier?

### Hypothesis

**Prediction (plan page, 2026-10-09):** the synced on-wafer window (`wafer_time_*`, max toc − min tic across PEs) is shorter
than the barrier-closed host time, by the launch and barrier costs. The flag-off build's stream time is within ±10% of S1
(43.5–48.9 ms). I also hoped the window would show a compute time of a few ms. That hope was mis-framed (see below).
**Falsifier:** window ≥ barrier time (the reference clock or the barrier is wrong), or a ratio that varies a lot between passes.

### Result

In [ ]:
d = D["d56"]
if d.empty:
    lib.no_data("2026-10-09_dan56.csv")
else:
    d = d.assign(arm=np.where(d.sync, "sync, loop 5", "flag off"))
    display(lib.summarize(d, ["arm"], "stream_ms", 1, 1).rename(columns=lambda c: c + " barrier ms" if c in ("min", "median", "max") else c))
    on = d[d.sync].assign(wafer_ms=lambda x: x.wafer_time_avg * 1e3, host_ms=lambda x: x.host_pass_time_avg * 1e3)
    display(on[["wafer_ms", "host_ms", "wafer_host_ratio_max"]].assign(wafer_min_ms=on.wafer_time_min * 1e3,
            wafer_max_ms=on.wafer_time_max * 1e3).round(2))
    print(f"median on-wafer window {on.wafer_ms.median():.1f} ms vs flag-off barrier {d[~d.sync].stream_ms.median():.1f} ms "
          f"-> ratio {on.wafer_ms.median() / d[~d.sync].stream_ms.median():.3f}")
    fig, axs = lib.panels(1, height=4.2)
    ax = axs[0]
    rows = [("flag off: barrier-closed", d[~d.sync].stream_ms, 0), ("sync: barrier-closed", on.stream_ms, 1),
            ("sync: on-wafer window", on.wafer_ms, 2)]
    for y, (lab, v, i) in enumerate(rows):
        st = lib.style("index", i)
        ax.scatter(v, [y] * len(v), s=60, color=st["color"], marker=st["marker"], edgecolors="white", zorder=3)
        ax.annotate(f"median {v.median():.1f} ms", (v.max(), y), xytext=(10, 0), textcoords="offset points", color=st["color"], va="center")
    ax.set_yticks(range(len(rows)), [r[0] for r in rows]); ax.set_ylim(-0.6, 2.6)
    lib.finish(ax, "time per SpMV pass (ms)", "", "65,536 PEs, n=3 runs per arm (5 passes each for sync)")
    ax.set_xlim(35, 52); plt.show()

### What that means

*(Written 2026-10-13.)*
- **The clocks agree.** The on-wafer window is 40.7 / 40.8 / 41.4 ms. That's **0.88 of the sync build's own barrier time in every one
  of the 15 passes**, and 0.90 of the flag-off build's 45.2 ms. The barrier overstates on-wafer time by a steady 11–13%: launch,
  start/stop skew across PEs, and the barrier itself.
- The sync build costs about +1 ms on the host metric (46.2 vs 45.2 ms).
- **The compute-time hope was mis-framed.** valtask computes while the values arrive, so the window spans the transfer. It can't
  isolate compute. That needs a resident-A run, the CG design.
- **Consequence:** the barrier-closed numbers from 10-07 onward can be trusted as an upper bound within 11–13%.

### What the vendor says

**Agrees.** [BW]:57–58 "To record the elapsed time on host may not show the true timing because the runtime may not start the
transaction when the user calls a H2D/D2H command", and the vendor benchmark "samples the timing on the device" ([BW]:60).
Synced timing vendors that sync module (csl-examples, `src/csl/sync/`).

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 3. Weak scaling of the stream: what limits it?

### Hypothesis

**Prediction (plan page):** bytes grow with PEs (~1 KB/PE), so time grows roughly linearly at 1.36–2.0 GB/s (the 65K and 10-08
full-wafer rates). Below 65K, fixed costs dominate. Full wafer 430–470 ms. Hope: the rate reaches ≥2 GB/s and flattens.
**Model tested afterwards (not pre-registered):** valtask serializes its W column transfers, so
T = W · (c + bytes_per_column / B). This was fitted once on all runs.
**Falsifier for the model:** residuals beyond run-to-run noise (~5–10%), or a size-dependent drift in them.

### Result

In [ ]:
if ws.empty:
    lib.no_data("2026-10-09_weak-scaling.csv")
else:
    ws["W"] = ws.grid_width
    ws["col_bytes"] = ws.h2d_bytes_total / ws.W
    ws["col_s"] = ws.stream_time_corrected / ws.W
    A = np.c_[np.ones(len(ws)), ws.col_bytes]
    (c, inv), *_ = np.linalg.lstsq(A, ws.col_s, rcond=None)
    B = 1 / inv
    ws["model_ms"] = ws.W * (c + ws.col_bytes / B) * 1e3
    print(f"fit over {len(ws)} runs: per-column fixed cost c = {c * 1e6:.1f} us, per-column bandwidth B = {B / 1e9:.2f} GB/s")
    t = ws.groupby("num_pes").agg(grid=("grid_width", lambda s: f"{s.iloc[0]}x{ws.loc[s.index[0], 'grid_height']}"),
        n=("stream_ms", "size"), MB=("h2d_bytes_total", lambda s: s.iloc[0] / 1e6), min_ms=("stream_ms", "min"),
        median_ms=("stream_ms", "median"), max_ms=("stream_ms", "max"), GBps=("stream_GBps", "median"), model_ms=("model_ms", "first"))
    t["resid %"] = 100 * (t.model_ms / t.median_ms - 1)
    t["fixed-cost share %"] = 100 * (t.index.map(ws.groupby("num_pes").W.first()) * c * 1e3) / t.median_ms
    display(t.round(2))
    fig, axs = lib.panels(2, sharex=False, sharey=False)
    ax = axs[0]
    ax.scatter(ws.col_bytes / 1e3, ws.col_s * 1e3, s=40, color=lib.BLUE, edgecolors="white", zorder=3, label="each run")
    xs = np.linspace(0, ws.col_bytes.max() * 1.05, 50)
    ax.plot(xs / 1e3, (c + xs / B) * 1e3, color=lib.ORANGE, ls="--", label=f"fit: {c * 1e6:.1f} µs + bytes / {B / 1e9:.2f} GB/s")
    for p, g in ws.groupby("num_pes"):
        ax.annotate(f"{p:,}", (g.col_bytes.iloc[0] / 1e3, g.col_s.median() * 1e3), xytext=(6, -12), textcoords="offset points", fontsize=10)
    lib.finish(ax, "bytes per column transfer (KB)", "time per column transfer (ms)", "one serialized column transfer")
    ax = axs[1]
    lib.draw_series(ax, ws, "num_pes", "stream_ms", "index", 0, label="measured (median line)")
    m = ws.groupby("num_pes").model_ms.first()
    ax.plot(m.index, m.values, color=lib.ORANGE, ls="--", marker="s", label="model")
    lib.finish(ax, "PEs", "stream time, barrier-closed (ms)", "weak scaling, ~200 NNZ/PE, d=1e-3", logx=True, logy=True,
               xticks=[1024, 4096, 16384, 65536, 262144, 857010], yticks=[1, 3, 10, 30, 100, 300, 1000])
    plt.show()

### What that means

*(Written 2026-10-13.)*
- **Every point validated, and the predictions held:** 262K 158 ms, 428K 249 ms, full wafer **458.8 ms** (417.7–480.1; 10-08's
  single run of 443 ms falls inside that range).
- **The rate rises with scale** (0.37 → 1.86 GB/s) and does not flatten at ≥2 GB/s. 13.1× the PEs from 65K costs 10.2× the time.
- **The model fits all 7 sizes within ~3%** (fit over all 21 runs) with one fixed cost per column (~57 µs) and one per-column bandwidth (~2.07 GB/s).
  The fixed cost explains the low small-size rates: it's 82% of the time at 1K, 32% at 65K and 9% at the full wafer.
  The ceiling the rate approaches is the per-column bandwidth, not the 12 GB/s the link delivers for copies (section 4).
- **What follows:** the lever is overlapping the column transfers. Header tolerates overlap. For valtask it's untested on hardware,
  and the simulator does not reproduce the hangs. Fewer grid columns would cut the fixed share.
- **Not established:** what sets the 2 GB/s per column (channel mapping of a 1-wide rectangle vs per-PE consumption). The model is
  descriptive, fitted after seeing the data.

### What the vendor says

**Partly comparable.** [BW]:19–20 "the overhead of TCP is about 200 us, a non-negligible cost when the transaction is small."
Our fixed cost per column transfer is ~57 µs, under that figure, but of the same kind. [SDK]:112 "You can block and unblock the input
tensor colors to overlap computation and communication": that's on-wafer overlap. **Silent** on overlapping host transfers, and on what a
1-column-wide streaming transfer can reach.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 4. Setup copy across scale: where is the ceiling?

### Hypothesis

**Prediction:** setup copy rises with transfer size and saturates near the M1/M4 figure of 10.3 GB/s at 16 channels.
**Falsifier:** saturation clearly above or below 10.3 GB/s.

### Result

In [ ]:
if ws.empty:
    lib.no_data("2026-10-09_weak-scaling.csv")
else:
    display(lib.summarize(ws, ["num_pes"], "setup_GBps", 1, 2).assign(MB=ws.groupby("num_pes").setup_h2d_bytes.first().values / 1e6))
    big = ws[ws.num_pes >= 65536]
    A = np.c_[np.ones(len(big)), big.setup_h2d_bytes]
    (c, inv), *_ = np.linalg.lstsq(A, big.setup_h2d_time_corrected, rcond=None)
    print(f"fit >= 65K PEs: setup = {c * 1e3:.2f} ms + bytes / {1 / inv / 1e9:.2f} GB/s, "
          f"max residual {100 * np.max(np.abs((A @ [c, inv]) / big.setup_h2d_time_corrected - 1)):.1f}%")
    fig, axs = lib.panels(1)
    ax = axs[0]
    lib.draw_series(ax, ws, "num_pes", "setup_GBps", "index", 0, label="setup copy, 32 strips")
    ax.axhline(10.3, color=lib.INK, ls=(0, (2, 3)), lw=1.2, label="M1/M4 figure, 10.3 GB/s")
    lib.draw_claims(ax, 16)
    lib.finish(ax, "PEs", "setup copy, barrier-closed (GB/s)", "16 channels, col-major, 32 strips", logx=True,
               xticks=[1024, 4096, 16384, 65536, 262144, 857010])
    ax.set_ylim(0, 16)
    lib.place_claims(fig); plt.show()

### What that means

*(Written 2026-10-13.)*
- **Falsified, upward:** setup saturates at **12.0 GB/s** from 262K on, not 10.3. The fit from 65K up is ~1.3 ms + bytes / 12.0 GB/s, within ~5% (the 65K runs scatter most).
- 12.0 GB/s is **96% of one 100 GbE link** (12.5 GB/s). The copy is bandwidth-bound, probably at the link itself.
- The 10.3 GB/s figure was measured on 65K-sized transfers. It's a size-dependent rate, not a ceiling. The paper corrections list was updated (f140bfd).
- **Not established:** whether more than one link is in use. The rate stopping just under one link's line rate suggests one.

### What the vendor says

**Consistent.** [BW]:17 "The host connects the device via 100Gbps ethernets." That's one link = 12.5 GB/s (red line; the plural
leaves the link count open). [BW]:18–19 "around 7Gbps" per channel: 14 GB/s at 16 channels, which we don't reach. The binding limit is
nearer the link than the channel sum.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 5. Did the wall time get better? Before vs after, by phase

### Hypothesis

**Prediction:** the optimizations (valtask, column mode, col-major setup, 16 channels, strips) cut wafer I/O by an order of
magnitude at 65K. Wall time improves much less, because program load (~40 s) and host tiling (~10 s) don't change.
**Falsifier:** wall time tracking wafer I/O.

### Result

In [ ]:
timed = ["matrix_generation_time", "vector_generation_time", "tile_extract_time", "sdk_init_time", "program_load_time",
         "setup_done_time", "setup_barrier_d2h_time", "setup_barrier_d2h_repeat_time", "device_done_time", "barrier_d2h_time",
         "barrier_d2h_repeat_time", "memcpy_d2h_time", "gather_results_time", "validation_time"]
def phases(d, label):
    d = d.copy()
    d["encode_s"] = d.total_time - d[[c for c in timed if c in d]].fillna(0).sum(axis=1)   # untimed: host stream building
    d["io_s"] = d.setup_h2d_time_corrected + d.stream_time_corrected
    r = d[["program_load_time", "tile_extract_time", "encode_s", "io_s", "stream_time_corrected", "setup_h2d_time_corrected", "total_time"]].median()
    r["n"] = len(d); r.name = label
    return r
o, s1, fw = (D[k][D[k].num_pes == p] if not D[k].empty else D[k] for k, p in (("old", 65536), ("s1", 65536), ("fw", 857010)))
rows = []
if not o.empty:
    rows.append(phases(o[(o.encoding == "header") & (o.h2d_mode == "full") & (o.channels == 1)], "65K old defaults: header/full/1ch/row (10-07)"))
if not s1.empty:
    rows.append(phases(s1[(s1.encoding == "valtask") & (s1.channels == 16)], "65K valtask/column/16ch/col, 1 strip (S1 10-07)"))
rows.append(phases(D["d56"][~D["d56"].sync], "65K new defaults + 32 strips (10-09)"))
if not fw.empty:
    rows.append(phases(fw[fw.encoding == "header"], "857K header/column/16ch/col, 1 strip (10-08)"))
rows.append(phases(D["weak"][D["weak"].num_pes == 857010], "857K new defaults + 32 strips (10-09)"))
t = pd.DataFrame(rows)
t.columns = ["load s", "tiling s", "encode s (untimed)", "wafer I/O s", "stream s", "setup s", "total s", "n"]
display(t.round(3))
print(f"65K wafer I/O old -> new: {t.iloc[0]['wafer I/O s'] / t.iloc[2]['wafer I/O s']:.1f}x; "
      f"wall {t.iloc[0]['total s'] / t.iloc[2]['total s']:.2f}x; full wafer wall header -> new: {t.iloc[3]['total s'] / t.iloc[4]['total s']:.2f}x")
fig, axs = lib.panels(2, sharex=False, sharey=False, height=3.4)
for ax, sel, title in [(axs[0], [0, 1, 2], "65,536 PEs"), (axs[1], [3, 4], "857,010 PEs (full wafer)")]:
    sub = t.iloc[sel]
    left = np.zeros(len(sub))
    for col, color, lab in [("load s", lib.BLUE, "program load"), ("tiling s", lib.ORANGE, "host tiling"),
                            ("encode s (untimed)", lib.GREEN, "host stream encoding (untimed)"), ("wafer I/O s", lib.INK, "wafer I/O")]:
        ax.barh(range(len(sub)), sub[col], left=left, color=color, edgecolor="white", label=lab)
        left += sub[col].values
    for i, v in enumerate(sub["total s"]):
        ax.annotate(f"{v:,.0f} s", (v, i), xytext=(6, 0), textcoords="offset points", va="center")
    ax.set_yticks(range(len(sub)), [s.split(" (")[0] for s in sub.index]); ax.invert_yaxis()
    lib.finish(ax, "wall time per run (s)", "", title)
    ax.get_legend().remove()
axs[1].legend(loc="lower right", ncol=1)
fig.tight_layout()
plt.show()

### What that means

*(Written 2026-10-13.)*
- **Wafer I/O got ~20× faster at 65K** (1.44 s → 0.074 s; stream 11.6×, setup 33×). At the full wafer, against the header run, it's 1.45×.
- **The biggest wall-time win was on the host:** header needs a Python stream-building step (`build_stream_buffer`, untimed in the CSV,
  so it appears as the gap). That costs ~96 s at 65K and **~1,210 s at the full wafer**. valtask's equivalent is ~1 s / ~8 s. So the
  full-wafer wall time fell **4.1×** (1,593 → 390 s).
- **At 65K the wall time improved only 1.4×,** because **program load doubled** (37–46 s → ~90 s). It was already bimodal within S1 on 10-07
  for the same config, so it's an appliance effect, not ours. At the old load time the 65K run would be ~58 s, 2.5× faster than the old defaults.
- **What remains:** program load (217 s) and host tiling (159 s) are 96% of a full-wafer run. Wafer I/O is under 0.3% of the wall time.
- **Not established:** that the whole gap is encoding (it also holds small untimed steps). The header/valtask difference, ~95 s at 65K
  and ~1,200 s at the full wafer, is far larger than any of those. Add a timer around `build_*_stream_buffer`.

### What the vendor says

**Silent.** [SDK]:7 says `SdkRuntime` is used to "load programs, launch kernels, and stream tensor data", but gives no load-time figures.
Host-side data preparation is the user's code.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 6. Accuracy across scale

### Hypothesis

**Prediction:** fp32 accumulation error grows roughly like √n (summing ~n·d random-sign terms per row); all runs stay below 1e-6.
**Falsifier:** growth faster than linear in n, or any run above 1e-6.

### Result

In [ ]:
if ws.empty:
    lib.no_data("weak scaling")
else:
    e = ws.groupby("matrix_rows").relative_error.agg(["size", "min", "max"])
    e["max / sqrt(n)"] = e["max"] / np.sqrt(e.index)
    display(e)
    k = e["max / sqrt(n)"].median()
    fig, axs = lib.panels(1, height=4.4)
    ax = axs[0]
    ax.scatter(ws.matrix_rows, ws.relative_error, s=50, color=lib.BLUE, edgecolors="white", zorder=3, label="each run")
    xs = np.array([1.2e4, 5e5]); ax.plot(xs, k * np.sqrt(xs), color=lib.ORANGE, ls="--", label=f"{k:.2e} · √n")
    lib.finish(ax, "matrix dimension n", "relative error vs fp64", "all 18 weak-scaling runs + 65K", logx=True, logy=True,
               xticks=[14300, 57200, 114000, 229000, 414000], yticks=[5e-8, 1e-7, 2e-7, 3e-7])
    plt.show()

### What that means

*(Written 2026-10-13.)*
- Error grows close to √n (the ratio max/√n stays within 4.3–5.2e-10) and peaks at **2.75e-07 at the full wafer**. Repeat runs at the
  same size give identical errors: the reduction order is deterministic.
- The 16-channel default, which clamps to the grid height below 16 rows (`4d7d244`), validated at every size.

### What the vendor says

**Silent.** No vendor figure for fp32 SpMV accuracy.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.